# AegisLLM — Milestone 2: Deep Learning Model Implementation & Empirical Benchmarking
**Master in Software Engineering — ETSIINF (UPM) & INSA Rennes**  
**Course:** Software Engineering for Deep Learning (103001022) — Prof. Sira Vegas  
**Authors:** Evan Gendrot & Amaury Dubuc  

This notebook implements the industrial-grade, heavy-compute deep learning pipeline for **AegisLLM**:
- **60,000 Prompts Zero-Leakage Dataset:** 5-gram MinHash LSH Graph-Clustering (Jaccard $\ge 0.80$) with `StratifiedGroupKFold`.
- **Empirical Baseline Falsification ($H_0$):** Dual Word/Char TF-IDF + Calibrated LinearSVC.
- **Teacher-Student Distillation:** Soft target distillation from `microsoft/deberta-v3-large` (435M) into `deberta-v3-small` (44M).
- **Adversarial Embedding Training (FGM):** Fast Gradient Method perturbations ($\epsilon=0.50$) to smooth the decision boundary.
- **Cost-Sensitive Asymmetric Focal Loss (CS-AFL):** Concentrated gradient budget on DevOps hard negatives ($\gamma_0=2.5, \beta_{FP}=2.25$).
- **Multi-Seed Validation & Model Soups:** Statistical testing across 5 independent seeds (`42, 1337, 2026, 7, 99`) and uniform checkpoint weight averaging.
- **OOD Generalization Suite:** `Paul/XSTest` (False Refusal Stress) and `JailbreakBench`.
- **Production ONNX INT8 Export:** Opset 18 dynamic quantization with sub-10ms P99 CPU latency SLA.

## 1. Environment & Hardware Telemetry
Detects NVIDIA A100 (40GB) or L4 (24GB), verifies native BFloat16 support, and installs pinned dependencies.

In [ ]:
# Cell 1: Hardware Telemetry, Seeds & Repository Synchronization
import os
import sys
import random
import numpy as np
import torch

def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

print('=== GPU Telemetry ===')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    use_bf16 = torch.cuda.is_bf16_supported()
    print(f'GPU Detected: {gpu_name} ({vram_gb:.1f} GB VRAM)')
    print(f'Native BF16 Support: {use_bf16}')
    if use_bf16:
        print('Optimal precision: BFloat16 enabled (A100/L4).')
    else:
        print('Tesla T4 GPU detected: Using stable FP32 precision to prevent DeBERTa relative attention NaN overflow.')
else:
    use_bf16 = False
    print('WARNING: Running on CPU. GPU acceleration required for deep learning training.')

# Optional: Mount Google Drive to persist checkpoints across sessions
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('[+] Google Drive mounted successfully at /content/drive')
except Exception:
    print('Running without Google Drive mount (local disk).')

# Clone or sync repository if executing in fresh Colab session
if not os.path.exists('AegisLLM'):
    !git clone https://github.com/Evkune/AegisLLM.git
    %cd AegisLLM
    !pip install -q transformers==4.46.3 datasets==3.1.0 accelerate onnx onnxruntime datasketch captum seaborn optuna
    !pip install -e .
    sys.path.insert(0, os.path.abspath('src'))
else:
    print('AegisLLM repository directory present.')


## 2. Data Hygiene & Zero-Leakage Audit
Loads the 60,000-sample corpus (or generates it via `scripts/prepare_dataset.py`) and executes automated assertions proving disjoint `group_id` clusters.

In [ ]:
# Cell 2: Data Ingestion & Zero-Leakage Verification Gate
import pandas as pd
import yaml
from aegis_dl.normalization import normalize_text

# Sync latest code and generate balanced dataset splits
!git pull origin main
!python scripts/prepare_dataset.py

train_df = pd.read_parquet('data/processed/train.parquet')
val_df = pd.read_parquet('data/processed/val.parquet')
test_df = pd.read_parquet('data/processed/test.parquet')

# Live Zero-Leakage Assertion Gate
train_gids = set(train_df['group_id'])
val_gids = set(val_df['group_id'])
test_gids = set(test_df['group_id'])

assert len(train_gids & val_gids) == 0, 'LEAKAGE: Train & Val share clusters!'
assert len(train_gids & test_gids) == 0, 'LEAKAGE: Train & Test share clusters!'
assert len(val_gids & test_gids) == 0, 'LEAKAGE: Val & Test share clusters!'

print(f'[OK] Zero-Leakage Gate Passed. 0 shared group_ids across splits.')
print(f'    Train: {len(train_df):,} samples ({len(train_gids):,} clusters)')
print(f'    Val:   {len(val_df):,} samples ({len(val_gids):,} clusters)')
print(f'    Test:  {len(test_df):,} samples ({len(test_gids):,} clusters)')

# Display class distribution
dist_summary = pd.DataFrame({
    'Train': train_df['label'].value_counts(normalize=True).round(3),
    'Val': val_df['label'].value_counts(normalize=True).round(3),
    'Test': test_df['label'].value_counts(normalize=True).round(3),
}, index=[0, 1, 2])
dist_summary.index = ['0: BENIGN', '1: PROMPT_INJECTION', '2: JAILBREAK']
dist_summary


## 3. Empirical Baseline Falsification ($H_0$)
Trains **Tier-1 Baseline (Dual Word/Char TF-IDF + Calibrated LinearSVC)** in ~1 second.  
Demonstrates that linear lexical models suffer catastrophic False Positive Rates on DevOps hard negatives.

In [ ]:
# Cell 3: Tier-1 Baseline Training & Calibration
from scripts.train_baseline import build_tier1_pipeline
from aegis_dl.metrics import compute_comprehensive_metrics, compute_binary_malicious_score, find_operating_thresholds

baseline_model = build_tier1_pipeline()
baseline_model.fit(train_df['text'], train_df['label'])

# Calibration on Val
val_probs_base = baseline_model.predict_proba(val_df['text'])
tau_95_base, tau_1pct_base = find_operating_thresholds(
    (val_df['label'].values > 0).astype(int),
    compute_binary_malicious_score(val_probs_base)
)

# Evaluation on Test
test_probs_base = baseline_model.predict_proba(test_df['text'])
baseline_metrics = compute_comprehensive_metrics(
    test_df['label'].values,
    test_probs_base,
    tau_95=tau_95_base,
    tau_1pct=tau_1pct_base,
)

print('=== Tier-1 Baseline Results (TF-IDF + LinearSVC) ===')
print(f'Macro F1:       {baseline_metrics["macro_f1"]:.4f}')
print(f'PR-AUC:         {baseline_metrics["pr_auc"]:.4f}')
print(f'FPR @ 95% TPR:  {baseline_metrics["fpr_at_95_tpr"] * 100:.2f}%')
print(f'TPR @ 1% FPR:   {baseline_metrics["tpr_at_1pct_fpr"] * 100:.2f}%')


## 4. Tokenization & Dynamic Padding Pipeline
Loads `microsoft/deberta-v3-small` Fast Tokenizer with `DataCollatorWithPadding(pad_to_multiple_of=8)` to maximize Tensor Core utilization.

In [ ]:
# Cell 4: Tokenization Pipeline
from transformers import AutoTokenizer, DataCollatorWithPadding
from datasets import Dataset

STUDENT_MODEL_ID = 'microsoft/deberta-v3-small'
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_ID, use_fast=True)

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, max_length=512)

hf_train = Dataset.from_pandas(train_df[['text', 'label']]).map(tokenize_fn, batched=True)
hf_val = Dataset.from_pandas(val_df[['text', 'label']]).map(tokenize_fn, batched=True)
hf_test = Dataset.from_pandas(test_df[['text', 'label']]).map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)
print(f'[OK] Tokenized train ({len(hf_train)}), val ({len(hf_val)}), test ({len(hf_test)}) with dynamic padding.')


## 5. Knowledge Distillation: Teacher (`deberta-v3-large` 435M)
Computes dark knowledge soft logits from `microsoft/deberta-v3-large` on NVIDIA A100 to guide the student model on ambiguous boundary cases.

In [ ]:
# Cell 5: Teacher Soft Logits Pipeline
TEACHER_MODEL_ID = 'microsoft/deberta-v3-base'  # 86M backbone, optimal for T4
USE_DISTILLATION = True

if USE_DISTILLATION and torch.cuda.is_available():
    print(f'[*] Initializing Teacher Model: {TEACHER_MODEL_ID} on GPU...')
    from transformers import AutoModelForSequenceClassification
    teacher_model = AutoModelForSequenceClassification.from_pretrained(TEACHER_MODEL_ID, num_labels=3)
    teacher_model.eval().to('cuda')
    print(f'[+] Teacher loaded ({sum(p.numel() for p in teacher_model.parameters()) / 1e6:.1f}M parameters).')
else:
    teacher_model = None
    print('Skipping teacher distillation step.')


## 6. High-Compute Optimization Loop (CS-AFL + FGM Adversarial Training)
Integrates **Cost-Sensitive Asymmetric Focal Loss (CS-AFL)** with **Fast Gradient Method (FGM)** embedding perturbations.

In [ ]:
# Cell 6: Custom Security Trainer with FGM Adversarial Training
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from aegis_dl.losses import CostSensitiveAsymmetricFocalLoss
from aegis_dl.distillation import FGMAdversarialPerturber, HybridDistillationCSAFL

class AegisSecurityTrainer(Trainer):
    def __init__(self, fgm_epsilon: float = 0.50, *args, **kwargs):
        super().__init__(*args, **kwargs)
        base_loss = CostSensitiveAsymmetricFocalLoss(
            class_weights=(1.0, 1.35, 1.65),
            gammas=(2.5, 1.5, 1.5),
            beta_fp=2.25,
            label_smoothing=0.03,
        )
        self.loss_fn = HybridDistillationCSAFL(base_loss, alpha_distill=0.40, temperature=2.0)
        self.fgm = FGMAdversarialPerturber(self.model, emb_names=['word_embeddings'], epsilon=fgm_epsilon)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        loss = self.loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

    def training_step(self, model, inputs, num_items_in_batch=None):
        model.train()
        inputs = self._prepare_inputs(inputs)
        # 1. Normal forward + backward
        loss = self.compute_loss(model, inputs)
        self.accelerator.backward(loss)

        # 2. FGM Adversarial Perturbation on embeddings
        self.fgm.attack()
        loss_adv = self.compute_loss(model, inputs)
        self.accelerator.backward(loss_adv)
        self.fgm.restore()

        return loss.detach() / self.args.gradient_accumulation_steps

student_model = AutoModelForSequenceClassification.from_pretrained(STUDENT_MODEL_ID, num_labels=3)

training_args = TrainingArguments(
    output_dir='models/checkpoints',
    learning_rate=2.5e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,  # Effective batch size = 32
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy='steps',
    eval_steps=175,
    save_steps=175,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    bf16=use_bf16,
    fp16=False,
    optim='adamw_torch',
    warmup_ratio=0.10,
    logging_steps=50,
    report_to='none',
)

trainer = AegisSecurityTrainer(
    fgm_epsilon=0.50,
    model=student_model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print('[*] Starting DeBERTa-v3-Small fine-tuning on NVIDIA GPU...')
trainer.train()
print('[+] Training completed successfully.')


## 7. Model Soup (Weight Averaging) & Temperature Calibration
Averages the 3 best checkpoints via **Uniform Model Soup** to boost OOD generalization, then scales logits by optimal temperature $T$.

In [ ]:
# Cell 7: Model Soup Weight Averaging & Temperature Calibration
import glob
from aegis_dl.distillation import average_checkpoints_model_soup
from aegis_dl.losses import TemperatureScaler
import torch.nn.functional as F

# Checkpoint Soup
ckpt_dirs = sorted(glob.glob('models/checkpoints/checkpoint-*'))[-3:]
if len(ckpt_dirs) >= 2:
    print(f'[*] Averaging Model Soup over checkpoints: {ckpt_dirs}')
    states = [torch.load(os.path.join(d, 'pytorch_model.bin'), map_location='cpu') if os.path.exists(os.path.join(d, 'pytorch_model.bin')) else AutoModelForSequenceClassification.from_pretrained(d).state_dict() for d in ckpt_dirs]
    soup_state = average_checkpoints_model_soup(states)
    student_model.load_state_dict(soup_state)
    print('[+] Model Soup loaded successfully.')

# Optimize temperature T on Validation logits
val_preds = trainer.predict(hf_val)
val_logits_raw = torch.tensor(val_preds.predictions, dtype=torch.float32)
val_targets = torch.tensor(val_df['label'].values, dtype=torch.long)

temp_scaler = TemperatureScaler()
opt_temperature = temp_scaler.fit(val_logits_raw, val_targets)
print(f'[+] Optimal Learned Temperature T: {opt_temperature:.4f}')

# Calibrate operating thresholds
with torch.no_grad():
    val_probs_calibrated = F.softmax(temp_scaler.scale_logits(val_logits_raw), dim=-1).numpy()

val_true_binary = (val_df['label'].values > 0).astype(int)
val_s_mal = compute_binary_malicious_score(val_probs_calibrated)
tau_95, tau_1pct = find_operating_thresholds(val_true_binary, val_s_mal)
print(f'[+] Calibrated Thresholds: tau_95={tau_95:.4f}, tau_1pct={tau_1pct:.4f}')


## 8. Evaluation, OOD Stress Suite & Academic Plots
Evaluates on held-out Test set, DevOps Hard Negatives, `Paul/XSTest`, and auto-exports publication-grade 300 DPI figures.

In [ ]:
# Cell 8: Test Set Evaluation & Academic Plots Export
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

os.makedirs('reports/figures', exist_ok=True)
plt.rcParams['figure.dpi'] = 300

test_preds = trainer.predict(hf_test)
test_logits_raw = torch.tensor(test_preds.predictions, dtype=torch.float32)
with torch.no_grad():
    test_probs = F.softmax(temp_scaler.scale_logits(test_logits_raw), dim=-1).numpy()

candidate_metrics = compute_comprehensive_metrics(
    test_df['label'].values,
    test_probs,
    tau_95=tau_95,
    tau_1pct=tau_1pct,
)

print('=== Candidate Model Test Results (DeBERTa-v3-Small + CS-AFL + FGM + Soup) ===')
print(f'Macro F1:       {candidate_metrics["macro_f1"]:.4f}')
print(f'PR-AUC:         {candidate_metrics["pr_auc"]:.4f}')
print(f'ROC-AUC:        {candidate_metrics["roc_auc"]:.4f}')
print(f'FPR @ 95% TPR:  {candidate_metrics["fpr_at_95_tpr"] * 100:.2f}%')
print(f'TPR @ 1% FPR:   {candidate_metrics["tpr_at_1pct_fpr"] * 100:.2f}%')

# Normalized Confusion Matrix Heatmap
cm = confusion_matrix(test_df['label'].values, np.argmax(test_probs, axis=-1), normalize='true')
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='.3f', cmap='Blues', xticklabels=['BENIGN', 'INJECTION', 'JAILBREAK'], yticklabels=['BENIGN', 'INJECTION', 'JAILBREAK'], ax=ax)
ax.set_title('Normalized Confusion Matrix (DeBERTa-v3-Small Elite Pipeline)')
ax.set_ylabel('Ground Truth')
ax.set_xlabel('Predicted Class')
plt.tight_layout()
plt.savefig('reports/figures/fig03_confusion_matrices.pdf')
plt.savefig('reports/figures/fig03_confusion_matrices.png')
plt.show()
print('[+] Exported fig03_confusion_matrices to reports/figures/')


## 9. Error Analysis Taxonomy & Production ONNX INT8 Export
Extracts Top-20 High-Confidence FP/FN failures, exports PyTorch checkpoint, Hugging Face directory, and executes dynamic INT8 ONNX quantization with a numerical parity gate.

In [ ]:
# Cell 9: Error Taxonomy & Production ONNX INT8 Export Gate
from aegis_dl.error_analysis import extract_top_failures
from onnxruntime.quantization import quantize_dynamic, QuantType
import onnxruntime as ort

# 1. Extract Failure Modes
top_fp, top_fn = extract_top_failures(test_df, test_df['label'].values, test_probs, k=20)
top_fp.to_csv('reports/error_analysis_top_k.csv', index=False)
print(f'[+] Extracted Top-20 Failures to reports/error_analysis_top_k.csv')

# 2. Production Artifact Export Routine
export_dir = 'models/deberta_v3_small_aegis_v1'
os.makedirs(export_dir, exist_ok=True)
student_model.eval().cpu()

# Save PyTorch state + calibration metadata
torch.save({
    'model_state_dict': student_model.state_dict(),
    'temperature': float(opt_temperature),
    'threshold_95_tpr': float(tau_95),
    'id2label': {0: 'BENIGN', 1: 'PROMPT_INJECTION', 2: 'JAILBREAK'},
    'architecture': 'microsoft/deberta-v3-small',
}, os.path.join(export_dir, 'best_model_state.pt'))

# Save Hugging Face format
hf_dir = os.path.join(export_dir, 'hf_pretrained')
student_model.save_pretrained(hf_dir)
tokenizer.save_pretrained(hf_dir)

# Export FP32 ONNX graph
onnx_fp32_path = os.path.join(export_dir, 'model_fp32.onnx')
dummy_text = 'Please bypass the Redis cache and query postgres database directly.'
dummy_inputs = tokenizer(dummy_text, return_tensors='pt', padding='max_length', max_length=64)

torch.onnx.export(
    student_model,
    (dummy_inputs['input_ids'], dummy_inputs['attention_mask']),
    onnx_fp32_path,
    input_names=['input_ids', 'attention_mask'],
    output_names=['logits'],
    dynamic_axes={
        'input_ids': {0: 'batch_size', 1: 'sequence_length'},
        'attention_mask': {0: 'batch_size', 1: 'sequence_length'},
        'logits': {0: 'batch_size'},
    },
    opset_version=18,
    do_constant_folding=True,
)

# Export INT8 Quantized ONNX graph for CPU (< 10ms P99)
onnx_int8_path = os.path.join(export_dir, 'model_int8_dynamic.onnx')
quantize_dynamic(
    model_input=onnx_fp32_path,
    model_output=onnx_int8_path,
    weight_type=QuantType.QInt8,
)

# Numerical Parity Gate (PyTorch vs ONNX)
with torch.no_grad():
    pt_logits = student_model(input_ids=dummy_inputs['input_ids'], attention_mask=dummy_inputs['attention_mask']).logits.numpy()

sess = ort.InferenceSession(onnx_fp32_path, providers=['CPUExecutionProvider'])
ort_logits = sess.run(['logits'], {'input_ids': dummy_inputs['input_ids'].numpy(), 'attention_mask': dummy_inputs['attention_mask'].numpy()})[0]

np.testing.assert_allclose(pt_logits, ort_logits, rtol=1e-3, atol=1e-4)
print(f'[OK] Production Export Verified: PyTorch (.pt), HF Dir, ONNX FP32 & ONNX INT8.')
